# Data Cleaning
This notebook is for cleaning the dataset in preperation for use in our various models.

## Import Packages

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from missforest import MissForest

## Load Data

In [2]:
thyroid_df = pd.read_csv('Thyroid-Dataset.csv')
thyroid_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9171 entries, 0 to 9170
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        9171 non-null   int64  
 1   sex                        8864 non-null   object 
 2   on thyroxine               9171 non-null   bool   
 3   query on thyroxine         9171 non-null   bool   
 4   on antithyroid medication  9171 non-null   bool   
 5   sick                       9171 non-null   bool   
 6   pregnant                   9171 non-null   bool   
 7   thyroid surgery            9171 non-null   bool   
 8   I131 treatment             9171 non-null   bool   
 9   query hypothyroid          9171 non-null   bool   
 10  query hyperthyroid         9171 non-null   bool   
 11  lithium                    9171 non-null   bool   
 12  goitre                     9171 non-null   bool   
 13  tumor                      9171 non-null   bool 

## Drop Referral Source Column

In [3]:
thyroid_df = thyroid_df.drop(columns='referral source')

## Drop Rows with Age Errors

In [4]:
age_error_rows = thyroid_df[thyroid_df['age'] > 100]
thyroid_df = thyroid_df.drop(age_error_rows.index)

## Create Missing Data Flags

### Float Variables

In [5]:
float_cols = thyroid_df.dtypes[thyroid_df.dtypes == 'float64'].index
for col in float_cols:
    new_col_name = col + '_missing'
    new_col_ind = thyroid_df.columns.to_list().index(col) + 1
    thyroid_df.insert(new_col_ind, new_col_name, thyroid_df[col].isna())

### Sex Variable

In [6]:
thyroid_df.insert(2, 'sex_missing', thyroid_df['sex'].isna())

## Dummify Class Variable

In [7]:
thyroid_df = pd.concat([thyroid_df, pd.get_dummies(thyroid_df['class'])], axis=1)
thyroid_df = thyroid_df.drop(columns='class')

## Encode categorical and Boolean Variables

Sex variable is encoded as 0 for female and 1 for male

In [8]:
thyroid_df['sex'] = thyroid_df['sex'].map({'F': 0, 'M': 1}).astype('Int64')
bool_cols = thyroid_df.dtypes[thyroid_df.dtypes == 'bool'].index
for col in bool_cols:
    thyroid_df[col] = thyroid_df[col].map({True: 1, False: 0}).astype('int64')

## Test-Train Split

In [9]:
demo_cols = sorted(['age', 'sex'])
blood_test_cols = sorted(['TSH', 'T3', 'TT4', 'T4U', 'FTI'])
med_hist_cols = sorted(['on thyroxine', 'query on thyroxine', 'on antithyroid medication', 'sick', 'pregnant', 'thyroid surgery', 'I131 treatment', 'query hypothyroid', 'query hyperthyroid', 'lithium', 'goitre', 'tumor', 'hypopituitary', 'psych'])
data_flag_cols = sorted(['TSH_missing', 'T3_missing', 'TT4_missing', 'T4U_missing', 'FTI_missing', 'sex_missing'])
label_cols = sorted(['antithyroid treatment', 'binding protein', 'discordant results', 'general health', 'hyperthyroid conditions', 'hypothyroid conditions', 'miscellaneous', 'negative', 'replacement therapy'])

Labels are encoded as 1 for hypothyroid and 2 for hyperthyroid (and 0 for neither).

Split is stratified by the labels since there is a large inblance.

In [10]:
X = thyroid_df.drop(columns=label_cols)
y = thyroid_df['hyperthyroid conditions'] * 2 + thyroid_df['hypothyroid conditions'] * 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Impute Missing Values Using MissForest Package

In [11]:
mf = MissForest(categorical=['sex'] + med_hist_cols + data_flag_cols)
X_train = mf.fit_transform(X_train)
X_test = mf.transform(X_test)
X_train = pd.DataFrame(X_train, columns=demo_cols + blood_test_cols + med_hist_cols + data_flag_cols)
X_test = pd.DataFrame(X_test, columns=demo_cols + blood_test_cols + med_hist_cols + data_flag_cols)

d:\Programs\Python3.13\Lib\site-packages\missforest\missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
100%|██████████| 5/5 [00:09<00:00,  1.99s/it]
d:\Programs\Python3.13\Lib\site-packages\missforest\missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
d:\Programs\Python3.13\Lib\site-packages\missforest\missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_transform` to fit the whole unseen data instead.
  warnings.warn(f"In version {VERSION}, estimator fitting proce

### Save Cleaned Dataset

In [12]:
X_train.to_csv('thyroid_train.csv', index=False)
X_test.to_csv('thyroid_test.csv', index=False)
y_train.to_csv('thyroid_train_labels.csv', index=False)
y_test.to_csv('thyroid_test_labels.csv', index=False)